In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, trim, upper, lower

spark = SparkSession.builder \
    .appName("SandboxSilverCustomers") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin_rafa") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin_rafa") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()
    
input_path = "s3a://bronze/olist/customers"
output_path = "s3a://silver/olist/customers"

print(f"Lendo dados da camada Bronze: {input_path}")

Lendo dados da camada Bronze: s3a://bronze/olist/customers


In [2]:
# Lendo os dados de Bronze
df_bronze = spark.read.parquet(input_path)

df_bronze.show(5, truncate=False)
df_bronze.printSchema()

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|data_processamento_bronze |
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |2026-06-25 03:38:11.795977|
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|09790                   |sao bernardo do campo|SP            |2026-06-25 03:38:11.795977|
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|01151                   |sao paulo            |SP            |2026-06-25 03:38:11.795977|
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9ac

In [3]:
#Tratamento dos dados
df_silver = df_bronze \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("customer_unique_id", trim(col("customer_unique_id"))) \
    .withColumn("customer_zip_code_prefix", trim(col("customer_zip_code_prefix"))) \
    .withColumn("customer_city", lower(trim(col("customer_city")))) \
    .withColumn("customer_state", upper(trim(col("customer_state")))) \
    .withColumn("dt_criacao_silver", current_timestamp())

df_silver.show(5, truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+--------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|data_processamento_bronze |dt_criacao_silver         |
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+--------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |2026-06-25 03:38:11.795977|2026-06-25 03:53:44.505477|
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|09790                   |sao bernardo do campo|SP            |2026-06-25 03:38:11.795977|2026-06-25 03:53:44.505477|
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|01151        

In [4]:
df_silver.write.mode("overwrite").parquet(output_path)
print("Dados gravados com sucesso na camada Silver!")

Dados gravados com sucesso na camada Silver!
